# Dante nanoGPT 🪶 — train a tiny GPT on the Divine Comedy

**This is an EDUCATIONAL notebook.** It walks through the whole pipeline of a
from-scratch, character-level, nanoGPT-style transformer trained on Dante's
*Divine Comedy*, so you can see end to end how a small LLM is built and trained:

1. Set up the environment (clone the repo, install dependencies)
2. **Prepare** the data (download + clean the text, build the char encoding)
3. **Train** the model on a GPU
4. **Sample** generated verses from the trained checkpoint

> Tip: in Colab, enable the GPU via **Runtime → Change runtime type → Hardware
> accelerator → GPU** (the free T4 is plenty for this).

## 0. Check the GPU

This should show your GPU (e.g. a Tesla T4). If it says no GPU, enable the GPU
runtime as noted above (training still works on CPU, just much slower).

In [ ]:
!nvidia-smi || echo 'No GPU detected (you can still run on CPU).'

## 1. Get the code

Clone the repository and move into it. (If you've already cloned it in this
session, the clone is skipped.)

In [ ]:
import os

REPO_URL = 'https://github.com/DanieleSeverin/dante-nanogpt.git'
if not os.path.isdir('dante-nanogpt'):
    !git clone $REPO_URL
%cd dante-nanogpt
!ls -la

## 2. Install dependencies

Colab already ships with PyTorch and NumPy, so this is usually quick.

In [ ]:
!pip install -q -r requirements.txt

## 3. Prepare the data

Download the Italian *Divina Commedia* from Project Gutenberg, clean it, build the
character-level vocabulary, and write the 90/10 train/val split. Run once.

In [ ]:
!python data/prepare.py

## 4. Save checkpoints to Google Drive (recommended)

Colab's local disk is **temporary** — if the session disconnects (you close the
tab or it times out), everything including the checkpoint is wiped. Mounting your
Google Drive and saving there means the best checkpoint **survives disconnects**,
so you can come back later and still have your trained model.

Running the cell below will pop up a Google authorization prompt — accept it.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

OUT_DIR = '/content/drive/MyDrive/dante-nanogpt/out'
os.makedirs(OUT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', OUT_DIR)

## 5. Train

These parameters are a reasonable "serious" run for the free Colab T4 GPU. Watch
the loss go down and the periodic generated samples become more Dante-like.
Feel free to tweak `max_iters`, `n_layer`, `n_embd`, `block_size`.

Notes:
- `--out_dir={OUT_DIR}` saves the best checkpoint to your Drive.
- `--compile=False`: `torch.compile` adds a long one-time pause with little
  benefit on the T4, so we leave it off. (`train.py` automatically uses
  `float16` on the T4, which is much faster than `bfloat16` there.)
- For a quick first test, lower `--max_iters` (e.g. 2000).

In [ ]:
!python train.py \
  --device=cuda \
  --n_layer=6 --n_head=6 --n_embd=384 \
  --block_size=256 --batch_size=64 \
  --max_iters=5000 --eval_interval=250 --sample_interval=500 \
  --learning_rate=3e-4 --compile=False \
  --out_dir={OUT_DIR}

## 6. Generate verses

Sample from the trained checkpoint on your Drive. Try your own `--start` prompt
and tune `--temperature` (lower = safer, higher = more creative) and `--top_k`.

In [ ]:
!python sample.py --out_dir={OUT_DIR} --num_samples=3 --max_new_tokens=500 --temperature=0.8 --top_k=40

In [ ]:
!python sample.py --out_dir={OUT_DIR} --start='Nel mezzo del cammin di nostra vita' --max_new_tokens=400

---
That's the whole loop: prepare → train → sample. To experiment, change the
training hyperparameters in step 5 and re-run steps 5–6. Have fun learning how
transformers work! 🪶